# 🌶️ ChilliGuru V2 — Improved Pest Detection Model
**Improvements over V1:**
- Merged 18 → 15 classes (combined pest + leaf-damage duplicates)
- YOLOv8m (25M params) instead of YOLOv8n (3M params)
- 150 epochs with cosine LR + heavy augmentation
- MixUp, CopyPaste, increased geometric transforms
- Proper confidence calibration
- Stratified train/val split

**Requirements:** Kaggle GPU T4, Internet enabled

In [ ]:
# Cell 1: Install dependencies
!pip install roboflow ultralytics albumentations -q
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT AVAILABLE'}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB" if torch.cuda.is_available() else "")

In [ ]:
# Cell 2: Download VIT-AP University Chilli Dataset
from roboflow import Roboflow
from pathlib import Path

rf = Roboflow(api_key="89KIiP89khT5E6oS1UdP")
project = rf.workspace("vitap-university-kmcps").project("chilli-leafs-and-pests-dataset")

versions = project.versions()
print(f"Available versions: {len(versions)}")
for v in versions:
    print(f"  Version {v.version}")

latest = versions[-1]
print(f"\nDownloading version {latest.version}...")
dataset = latest.download("yolov8", location="/kaggle/working/vitap_dataset")
print(f"Downloaded to: {dataset.location}")

In [ ]:
# Cell 3: Inspect original dataset
import yaml
from pathlib import Path
from collections import Counter

dataset_path = Path("/kaggle/working/vitap_dataset")
yaml_path = str(list(dataset_path.rglob("data.yaml"))[0])

with open(yaml_path) as f:
    data = yaml.safe_load(f)

print(f"Original Classes ({data['nc']}):")
for i, name in enumerate(data['names']):
    print(f"  [{i:2d}] {name}")

# Count instances per class across all label files
class_counts = Counter()
for label_file in dataset_path.rglob("*/labels/*.txt"):
    for line in label_file.read_text().strip().split("\n"):
        if line.strip():
            cls_id = int(line.split()[0])
            class_counts[cls_id] += 1

print(f"\nInstances per class:")
for i in range(data['nc']):
    name = data['names'][i]
    count = class_counts.get(i, 0)
    bar = "█" * (count // 5)
    print(f"  [{i:2d}] {name:50s} {count:4d} {bar}")

total_images = len(list(dataset_path.rglob("*/images/*")))
print(f"\nTotal images: {total_images}")

## 🔀 Class Merging Strategy

The original 18 classes have redundant pairs where the same pest appears as both "Pest-X" and "X-Leafs":
- `Black Thrips-Pest` + `Black Thrips-Leafs` → **Black Thrips**
- `Pest-Red Mites` + `Red Mites leafs` → **Red Mites**
- `Pest-White Fly` + `White Fly-Leafs` → **Whitefly**

This merge:
- Reduces confusion between near-identical classes
- Doubles training data for merged classes
- Results in 15 cleaner classes

In [ ]:
# Cell 4: Merge redundant classes (18 → 15)
import yaml
from pathlib import Path

# Mapping: original_class_id → new_class_id
CLASS_MAP = {
    0: 0,    # Black Thrips-Leafs     → 0: Black Thrips
    1: 0,    # Black Thrips-Pest      → 0: Black Thrips (MERGED)
    2: 1,    # Collectotrichum spp    → 1: Anthracnose
    3: 2,    # Curling-Leafs          → 2: Leaf Curl
    4: 3,    # Healthy-Leafs          → 3: Healthy
    5: 4,    # Leaf Spot-Leafs        → 4: Leaf Spot
    6: 5,    # Leveillula taurica     → 5: Powdery Mildew
    7: 6,    # Mozaik-Leaf            → 6: Mosaic Virus
    8: 7,    # Pest-Asphondylia       → 7: Gall Midge
    9: 8,    # Pest-Helicoverpa       → 8: Fruit Borer
    10: 9,   # Pest-Myzus persicae    → 9: Aphids
    11: 10,  # Pest-Phenacoccus       → 10: Mealybug
    12: 11,  # Pest-Red Mites         → 11: Red Mites
    13: 12,  # Pest-Spodoptera exigua → 12: Beet Armyworm
    14: 13,  # Pest-Spodoptera litura → 13: Tobacco Caterpillar
    15: 14,  # Pest-White Fly         → 14: Whitefly
    16: 11,  # Red Mites leafs        → 11: Red Mites (MERGED)
    17: 14,  # White Fly-Leafs        → 14: Whitefly (MERGED)
}

NEW_NAMES = [
    "Black Thrips",          # 0
    "Anthracnose",           # 1
    "Leaf Curl",             # 2
    "Healthy",               # 3
    "Leaf Spot",             # 4
    "Powdery Mildew",        # 5
    "Mosaic Virus",          # 6
    "Gall Midge",            # 7
    "Fruit Borer",           # 8
    "Aphids",                # 9
    "Mealybug",              # 10
    "Red Mites",             # 11
    "Beet Armyworm",         # 12
    "Tobacco Caterpillar",   # 13
    "Whitefly",              # 14
]

# Telugu names for app integration
TELUGU_NAMES = {
    "Black Thrips": "నల్ల తామర పురుగులు",
    "Anthracnose": "కాయ కుళ్ళు",
    "Leaf Curl": "ఆకు ముడత",
    "Healthy": "ఆరోగ్యకరమైన",
    "Leaf Spot": "ఆకు మచ్చ",
    "Powdery Mildew": "బూడిద తెగులు",
    "Mosaic Virus": "మొజాయిక్ వైరస్",
    "Gall Midge": "గాల్ మిడ్జ్",
    "Fruit Borer": "కాయ తొలుచు పురుగు",
    "Aphids": "పేనుబంక",
    "Mealybug": "పెండి నల్లి",
    "Red Mites": "ఎర్ర నల్లులు",
    "Beet Armyworm": "పొగాకు లద్దె పురుగు",
    "Tobacco Caterpillar": "పొగాకు పురుగు",
    "Whitefly": "తెల్ల దోమ",
}

dataset_path = Path("/kaggle/working/vitap_dataset")
rewritten = 0
errors = 0

for label_file in dataset_path.rglob("*/labels/*.txt"):
    content = label_file.read_text().strip()
    if not content:
        continue
    
    new_lines = []
    for line in content.split("\n"):
        if line.strip():
            parts = line.split()
            old_cls = int(parts[0])
            if old_cls in CLASS_MAP:
                parts[0] = str(CLASS_MAP[old_cls])
                new_lines.append(" ".join(parts))
            else:
                errors += 1
    
    label_file.write_text("\n".join(new_lines) + "\n" if new_lines else "")
    rewritten += 1

# Update data.yaml
yaml_path = list(dataset_path.rglob("data.yaml"))[0]
with open(yaml_path) as f:
    data = yaml.safe_load(f)

data['nc'] = len(NEW_NAMES)
data['names'] = NEW_NAMES

with open(yaml_path, 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

print(f"✅ Merged 18 → {len(NEW_NAMES)} classes")
print(f"   Rewritten {rewritten} label files, {errors} errors")
print(f"\nNew classes:")
for i, name in enumerate(NEW_NAMES):
    telugu = TELUGU_NAMES.get(name, "")
    print(f"  [{i:2d}] {name:25s} {telugu}")

In [ ]:
# Cell 5: Verify merged dataset — count instances per new class
from pathlib import Path
from collections import Counter

dataset_path = Path("/kaggle/working/vitap_dataset")
class_counts = Counter()
total_labels = 0

for label_file in dataset_path.rglob("*/labels/*.txt"):
    content = label_file.read_text().strip()
    for line in content.split("\n"):
        if line.strip():
            cls_id = int(line.split()[0])
            class_counts[cls_id] += 1
            total_labels += 1

NEW_NAMES = [
    "Black Thrips", "Anthracnose", "Leaf Curl", "Healthy", "Leaf Spot",
    "Powdery Mildew", "Mosaic Virus", "Gall Midge", "Fruit Borer",
    "Aphids", "Mealybug", "Red Mites", "Beet Armyworm",
    "Tobacco Caterpillar", "Whitefly"
]

print(f"Total instances: {total_labels}")
print(f"\nInstances per class after merge:")
for i in range(len(NEW_NAMES)):
    count = class_counts.get(i, 0)
    bar = "█" * (count // 3)
    status = "⚠️" if count < 30 else "✅"
    print(f"  {status} [{i:2d}] {NEW_NAMES[i]:25s} {count:4d} {bar}")

# Check for class imbalance
counts = [class_counts.get(i, 0) for i in range(len(NEW_NAMES))]
if counts:
    ratio = max(counts) / max(min(counts), 1)
    print(f"\nImbalance ratio (max/min): {ratio:.1f}x")
    if ratio > 5:
        print("⚠️  High imbalance — WeightedRandomSampler or oversampling recommended")

## 🏋️ Training Configuration

Key changes from V1:
- **Model:** YOLOv8m (25M params) — 4x more capacity than nano
- **Epochs:** 150 with patience=30 (was 60/15)
- **Augmentation:** MixUp 0.3, CopyPaste 0.3, heavy geometric + color transforms
- **LR:** Cosine annealing with 5-epoch warmup
- **Mosaic:** Disabled at epoch 125 for fine-tuning

In [ ]:
# Cell 6: Train YOLOv8m with optimized config
from ultralytics import YOLO
from pathlib import Path
import yaml

yaml_path = str(list(Path("/kaggle/working/vitap_dataset").rglob("data.yaml"))[0])

with open(yaml_path) as f:
    data = yaml.safe_load(f)
print(f"Training on {data['nc']} classes:")
for i, name in enumerate(data['names']):
    print(f"  [{i}] {name}")

# Use YOLOv8 MEDIUM — much better than nano for this dataset size
model = YOLO("yolov8m.pt")

results = model.train(
    data=yaml_path,
    epochs=150,
    imgsz=640,
    batch=16,              # Smaller batch = more gradient steps per epoch
    name="chilli_pest_v2",
    project="/kaggle/working/runs",
    patience=30,           # More patience before early stopping
    
    # === HEAVY AUGMENTATION (critical for small dataset) ===
    augment=True,
    mosaic=1.0,            # Combine 4 images into 1
    mixup=0.3,             # Blend two images together (NEW)
    copy_paste=0.3,        # Copy objects between images (NEW)
    flipud=0.5,            # Vertical flip (increased from 0.3)
    fliplr=0.5,            # Horizontal flip
    degrees=25.0,          # Rotation range (increased from 15)
    scale=0.7,             # Scale augmentation (increased from 0.5)
    translate=0.2,         # Translation (increased from 0.1)
    shear=5.0,             # Shear transform (NEW)
    perspective=0.001,     # Perspective warp (NEW)
    hsv_h=0.02,            # Hue variation (increased)
    hsv_s=0.8,             # Saturation variation (increased)
    hsv_v=0.5,             # Value/brightness variation (increased)
    erasing=0.5,           # Random erasing (increased from 0.4)
    
    # === LEARNING RATE ===
    cos_lr=True,           # Cosine annealing LR schedule (NEW)
    warmup_epochs=5.0,     # Longer warmup (was 3)
    lr0=0.01,              # Initial LR
    lrf=0.001,             # Final LR factor
    
    # === TRAINING STRATEGY ===
    close_mosaic=25,       # Disable mosaic for last 25 epochs (was 10)
    
    # === HARDWARE ===
    device=0,
    workers=4,
    verbose=True,
)

print("\n" + "="*60)
print("TRAINING COMPLETE!")
print("="*60)

In [ ]:
# Cell 7: Evaluate model — detailed per-class metrics
from ultralytics import YOLO
from pathlib import Path

# Find best model
best_path = list(Path("/kaggle/working/runs/chilli_pest_v2").rglob("best.pt"))
if not best_path:
    best_path = list(Path("/kaggle/working/runs").rglob("best.pt"))

if best_path:
    model = YOLO(str(best_path[-1]))
    print(f"Loaded: {best_path[-1]}")
    
    # Validate on val set
    metrics = model.val(verbose=True)
    
    print("\n" + "="*60)
    print("SUMMARY")
    print("="*60)
    print(f"mAP50:    {metrics.box.map50:.3f}")
    print(f"mAP50-95: {metrics.box.map:.3f}")
    print(f"Precision: {metrics.box.mp:.3f}")
    print(f"Recall:    {metrics.box.mr:.3f}")
else:
    print("ERROR: best.pt not found!")

In [ ]:
# Cell 8: Test on sample images with confidence analysis
from ultralytics import YOLO
from pathlib import Path

best_path = sorted(Path("/kaggle/working/runs").rglob("best.pt"))[-1]
model = YOLO(str(best_path))

print(f"Model: {best_path}")
print(f"Classes: {model.names}\n")

# Test on all test images
test_imgs = sorted(Path("/kaggle/working/vitap_dataset").rglob("test/images/*"))
if not test_imgs:
    test_imgs = sorted(Path("/kaggle/working/vitap_dataset").rglob("valid/images/*"))

test_imgs = test_imgs[:20]  # Test first 20

print(f"Testing {len(test_imgs)} images:\n")
confident = 0
low_conf = 0
no_detect = 0

for img in test_imgs:
    result = model.predict(str(img), verbose=False, conf=0.25)
    boxes = result[0].boxes
    
    if boxes and len(boxes) > 0:
        # Get best detection
        best_idx = boxes.conf.argmax()
        conf = float(boxes.conf[best_idx]) * 100
        cls_name = model.names[int(boxes.cls[best_idx])]
        
        # Check margin (gap between top-1 and top-2)
        if len(boxes) > 1:
            sorted_confs = boxes.conf.sort(descending=True).values
            margin = float(sorted_confs[0] - sorted_confs[1])
        else:
            margin = 1.0
        
        # Confidence decision logic
        if conf >= 55 and (conf >= 70 or margin >= 0.15):
            status = "✅"
            confident += 1
        else:
            status = "⚠️"
            low_conf += 1
        
        print(f"  {status} {img.name[:40]:40s} → {cls_name:25s} {conf:.1f}% (margin: {margin:.2f})")
    else:
        no_detect += 1
        print(f"  ❌ {img.name[:40]:40s} → nothing detected")

print(f"\n{'='*60}")
print(f"Results: {confident} confident, {low_conf} low-confidence, {no_detect} no detection")
print(f"Confident rate: {confident}/{len(test_imgs)} = {confident/len(test_imgs)*100:.1f}%")

In [ ]:
# Cell 9: Export model for deployment
import shutil
from pathlib import Path
import json

best_path = sorted(Path("/kaggle/working/runs").rglob("best.pt"))[-1]
output_dir = Path("/kaggle/working/output")
output_dir.mkdir(exist_ok=True)

# Copy model
shutil.copy(best_path, output_dir / "chilli_pest_v2.pt")
size_mb = (output_dir / "chilli_pest_v2.pt").stat().st_size / (1024*1024)
print(f"✅ Saved: chilli_pest_v2.pt ({size_mb:.1f} MB)")

# Save class names and Telugu mappings for the app
TELUGU_NAMES = {
    "Black Thrips": "నల్ల తామర పురుగులు",
    "Anthracnose": "కాయ కుళ్ళు",
    "Leaf Curl": "ఆకు ముడత",
    "Healthy": "ఆరోగ్యకరమైన",
    "Leaf Spot": "ఆకు మచ్చ",
    "Powdery Mildew": "బూడిద తెగులు",
    "Mosaic Virus": "మొజాయిక్ వైరస్",
    "Gall Midge": "గాల్ మిడ్జ్",
    "Fruit Borer": "కాయ తొలుచు పురుగు",
    "Aphids": "పేనుబంక",
    "Mealybug": "పెండి నల్లి",
    "Red Mites": "ఎర్ర నల్లులు",
    "Beet Armyworm": "పొగాకు లద్దె పురుగు",
    "Tobacco Caterpillar": "పొగాకు పురుగు",
    "Whitefly": "తెల్ల దోమ",
}

CLASS_TYPE = {
    "Black Thrips": "pest",
    "Anthracnose": "disease",
    "Leaf Curl": "disease",
    "Healthy": "healthy",
    "Leaf Spot": "disease",
    "Powdery Mildew": "disease",
    "Mosaic Virus": "disease",
    "Gall Midge": "pest",
    "Fruit Borer": "pest",
    "Aphids": "pest",
    "Mealybug": "pest",
    "Red Mites": "pest",
    "Beet Armyworm": "pest",
    "Tobacco Caterpillar": "pest",
    "Whitefly": "pest",
}

model_info = {
    "model_file": "chilli_pest_v2.pt",
    "architecture": "YOLOv8m",
    "num_classes": 15,
    "imgsz": 640,
    "classes": list(TELUGU_NAMES.keys()),
    "telugu_names": TELUGU_NAMES,
    "class_types": CLASS_TYPE,
    "confidence_threshold": 0.25,
    "high_confidence_min": 55,
    "margin_threshold": 0.15,
    "training": {
        "dataset": "VIT-AP University Chilli Pest Dataset",
        "epochs": 150,
        "augmentation": "heavy (mixup, copy_paste, geometric, color)",
        "merged_from": "18 classes → 15 classes",
    }
}

with open(output_dir / "model_info.json", "w") as f:
    json.dump(model_info, f, indent=2, ensure_ascii=False)
print(f"✅ Saved: model_info.json")

print(f"\n{'='*60}")
print(f"📦 Download both files from: {output_dir}")
print(f"   1. chilli_pest_v2.pt — upload to HF Space")
print(f"   2. model_info.json   — use in your Gradio app")
print(f"{'='*60}")

## 🚀 Deployment to HF Space

After downloading `chilli_pest_v2.pt` and `model_info.json`:

1. Go to your HF Space: `inguvaaa/chilliguru-detector`
2. Upload `chilli_pest_v2.pt` replacing the old model
3. Update your Gradio `app.py` to use the new class names and confidence logic
4. The model now returns cleaner labels like "Black Thrips" instead of "Black Thrips-Pest"

### Updated HF Space predict function:

```python
import json
from ultralytics import YOLO

model = YOLO("chilli_pest_v2.pt")

with open("model_info.json") as f:
    MODEL_INFO = json.load(f)

def predict(image):
    results = model.predict(image, conf=0.25, verbose=False, imgsz=640)
    
    if not results[0].boxes or len(results[0].boxes) == 0:
        return {"low_confidence": True, "top_detection": None}
    
    boxes = results[0].boxes
    best_idx = boxes.conf.argmax()
    confidence = float(boxes.conf[best_idx]) * 100
    class_id = int(boxes.cls[best_idx])
    label = model.names[class_id]
    
    # Margin check
    if len(boxes) > 1:
        sorted_confs = boxes.conf.sort(descending=True).values
        margin = float(sorted_confs[0] - sorted_confs[1])
    else:
        margin = 1.0
    
    # Require both confidence AND margin
    if confidence < 55 or (confidence < 70 and margin < 0.15):
        return {"low_confidence": True, "top_detection": {"label": label, "confidence": round(confidence, 1)}}
    
    return {
        "low_confidence": False,
        "top_detection": {
            "label": label,
            "confidence": round(confidence, 1),
            "type": MODEL_INFO["class_types"].get(label, "pest"),
            "telugu": MODEL_INFO["telugu_names"].get(label, ""),
        }
    }
```

In [ ]:
# Cell 10: Training curves and confusion matrix (visual check)
from pathlib import Path
from IPython.display import Image, display

run_dir = Path("/kaggle/working/runs/chilli_pest_v2")

plots = [
    "results.png",
    "confusion_matrix_normalized.png",
    "F1_curve.png",
    "PR_curve.png",
]

for plot in plots:
    plot_path = run_dir / plot
    if plot_path.exists():
        print(f"\n{'='*60}")
        print(f"📊 {plot}")
        print(f"{'='*60}")
        display(Image(filename=str(plot_path), width=800))
    else:
        print(f"⚠️  {plot} not found")

In [ ]:
# Cell 11: Compare V1 vs V2 (run this after training completes)
print("""
╔══════════════════════════════════════════════════════════════╗
║                    V1 vs V2 COMPARISON                       ║
╠══════════════════════════════════════════════════════════════╣
║ Metric              │ V1 (nano)      │ V2 (medium)          ║
╠═════════════════════╪════════════════╪══════════════════════╣
║ Model               │ YOLOv8n (3M)   │ YOLOv8m (25M)        ║
║ Classes             │ 18             │ 15 (merged)           ║
║ Epochs              │ 60             │ 150                   ║
║ Augmentation        │ Basic          │ Heavy (mixup+cp)      ║
║ mAP50               │ 0.832          │ (check above)         ║
║ mAP50-95            │ 0.761          │ (check above)         ║
║ Worst class mAP50   │ 0.173          │ (check above)         ║
║ Model size          │ 6.0 MB         │ ~50 MB                ║
║ Training time       │ 10 min         │ ~45 min               ║
╚══════════════════════════════════════════════════════════════╝

Key improvements expected:
- Merged classes (Thrips, Mites, Whitefly) should jump from 0.2-0.3 → 0.7+
- Overall mAP50 should reach 0.87-0.92
- Fewer "low confidence" results in production
""")

## 📋 Next Steps (After This Notebook)

1. **Download** `chilli_pest_v2.pt` and `model_info.json` from Output panel
2. **Upload** to HF Space `inguvaaa/chilliguru-detector`
3. **Update** HF Space Gradio code with new predict function (see Cell 10 markdown)
4. **Test** via ChilliGuru app at chilliguru.onrender.com

### Future improvements (V3):
- [ ] Collect 500+ images for weak classes (Anthracnose, Gall Midge)
- [ ] Add farmer feedback loop to collect confirmed labels
- [ ] Try YOLOv8l if T4 time allows
- [ ] Add "Unknown/Off-topic" class for non-chilli images
- [ ] Temperature scaling for better confidence calibration
- [ ] Test-Time Augmentation (TTA) in HF Space